# Qwen Policy Dev (Option A reboot of exp004)

Smoke-tests `agents.qwen_policy_agent.QwenPolicyAgent` (D5/D6 work, post POSTMORTEM).
Five fixes vs original `agents/qwen_agent.py`:
F1 sampling+state-graph dedup, F2 vLLM prefix cache OR upscale=4, F3 StateGraph wrap, F4 change-feedback in prompt, F5 ACTION6 path.

This kernel does NOT submit. It writes `qwen_policy_dev.json` with per-game stats (levels_completed, mean s/action, change-rate) for ls20+vc33+ft09.

In [ ]:
import os, sys, time, json, subprocess
from pathlib import Path

# --- 1. Locate the mounted Qwen weights -------------------------------------
_CANDIDATES = [
    Path('/kaggle/input/qwen3-6-35b-a3b-bf16'),
    Path('/kaggle/input/datasets/cataluna84/qwen3-6-35b-a3b-bf16'),
]
QWEN_DIR = next((c for c in _CANDIDATES if c.exists() and (c / 'model.safetensors.index.json').exists()), None)
if QWEN_DIR is None:
    for p in Path('/kaggle/input').rglob('model.safetensors.index.json'):
        QWEN_DIR = p.parent
        break
if QWEN_DIR is None:
    print('[ERROR] Qwen dataset not mounted; tree:')
    subprocess.run(['find', '/kaggle/input', '-maxdepth', '3', '-type', 'd'])
    sys.exit(1)

print('Qwen weights at', QWEN_DIR)
print('Total size:', round(sum(f.stat().st_size for f in QWEN_DIR.rglob('*') if f.is_file())/1e9, 1), 'GB')

# --- 2. Install arc-agi SDK + Pillow from competition wheels ---------------
_WHEEL_CANDS = [
    Path('/kaggle/input/arc-prize-2026-arc-agi-3/arc_agi_3_wheels'),
    Path('/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels'),
]
WHEELS = next((c for c in _WHEEL_CANDS if c.exists()), _WHEEL_CANDS[0])
if WHEELS.exists():
    PILLOW_TARGET = '/kaggle/working/_pillow_pkg'
    Path(PILLOW_TARGET).mkdir(parents=True, exist_ok=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index',
                    f'--find-links={WHEELS}', '--upgrade', '--target', PILLOW_TARGET,
                    'pillow'], capture_output=True, text=True)
    if PILLOW_TARGET not in sys.path:
        sys.path.insert(0, PILLOW_TARGET)
    for mod in [m for m in list(sys.modules) if m.startswith('PIL')]:
        del sys.modules[mod]
    try:
        import PIL, PIL.ImageText  # noqa
        print(f'PIL ok: {PIL.__version__}')
    except Exception as e:
        print(f'[WARN] PIL: {type(e).__name__}: {e}')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-index',
                    f'--find-links={WHEELS}', 'arc-agi', 'arcengine'],
                   capture_output=True, text=True)
    print('arc-agi installed')
else:
    print('[WARN] no wheels at', WHEELS)

# --- 2b. Install transformers 5.7 (needed if vLLM falls back) ----------------
_TX_CANDS = [
    Path('/kaggle/input/arc-agi-3-transformers-wheels'),
    Path('/kaggle/input/datasets/cataluna84/arc-agi-3-transformers-wheels'),
]
TX_WHEELS = next((c for c in _TX_CANDS if c.exists()), None)
if TX_WHEELS is None:
    for p in Path('/kaggle/input').rglob('transformers-*.whl'):
        TX_WHEELS = p.parent
        break
if TX_WHEELS is not None:
    TX_TARGET = '/kaggle/working/_transformers_pkg'
    Path(TX_TARGET).mkdir(parents=True, exist_ok=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index',
                    f'--find-links={TX_WHEELS}', '--upgrade', '--target', TX_TARGET,
                    '--no-deps', 'transformers', 'tokenizers', 'accelerate',
                    'huggingface_hub', 'safetensors',
                    'regex', 'filelock', 'fsspec', 'pyyaml', 'tqdm'],
                   capture_output=True, text=True)
    if TX_TARGET not in sys.path:
        sys.path.insert(0, TX_TARGET)
    for mod in [m for m in list(sys.modules)
                if m.split('.')[0] in {'transformers', 'tokenizers', 'accelerate', 'huggingface_hub', 'safetensors'}]:
        del sys.modules[mod]
    try:
        import transformers as _tx
        print(f'transformers ok: {_tx.__version__}')
    except Exception as e:
        print(f'[WARN] transformers: {type(e).__name__}: {e}')
else:
    print('[WARN] transformers wheels not mounted')

# --- 2c. Probe whether vLLM is available on this Kaggle image ---------------
VLLM_AVAILABLE = False
try:
    import vllm  # noqa
    VLLM_AVAILABLE = True
    print(f'vllm pre-installed: {vllm.__version__}')
except ImportError:
    print('[INFO] vllm not present; will fall back to transformers')

# --- 3. Make local agents/ package importable ------------------------------
AGENTS_PKG = Path('/kaggle/working/agents')
if not AGENTS_PKG.exists():
    _AGENT_CANDS = [
        Path('/kaggle/input/arc-agi-3-agents-pkg/agents'),
        Path('/kaggle/input/datasets/cataluna84/arc-agi-3-agents-pkg/agents'),
    ]
    src = next((c for c in _AGENT_CANDS if c.exists()), None)
    if src is None:
        for p in Path('/kaggle/input').rglob('qwen_policy_agent.py'):
            src = p.parent
            break
    if src is None:
        for p in Path('/kaggle/input').rglob('qwen_agent.py'):
            src = p.parent
            break
    if src is not None:
        import shutil
        shutil.copytree(src, AGENTS_PKG)
        print(f'copied agents pkg from {src}')
    else:
        print('[ERROR] agents pkg not found')
        sys.exit(1)
sys.path.insert(0, '/kaggle/working')
for mod in [m for m in list(sys.modules) if m == 'agents' or m.startswith('agents.')]:
    del sys.modules[mod]
for f in sorted(Path(AGENTS_PKG).iterdir()):
    if f.suffix == '.py':
        print(f'  agents/{f.name}: {f.stat().st_size} B')

# --- 3b. Monkey-patch Qwen3.5-MoE GatedDeltaNet init (CUDA H100 workaround)
# transformers 5.7.0 calls Qwen3_5MoeForConditionalGeneration._init_weights
# on every submodule lacking _is_hf_initialized. For GatedDeltaNet's A_log
# parameter, the init triggers `.uniform_(0,16).log_()` which fails on the
# current Kaggle H100 image with: AcceleratorError: 'no kernel image is
# available for execution on the device' (H100 + recent torch wheels mismatch).
# Since the safetensors already contain valid weights (the load report shows
# the GatedDeltaNet keys as UNEXPECTED meaning we have them in safetensors),
# we no-op the init for those modules. v9 of the original D2 dev kernel did
# not hit this -- assumed Kaggle has since updated the image.
try:
    from transformers.models.qwen3_5_moe import modeling_qwen3_5_moe as _qmm
    _orig_init = _qmm.Qwen3_5MoeForConditionalGeneration._init_weights
    def _patched_init(self, module):
        if 'GatedDeltaNet' in type(module).__name__:
            return
        return _orig_init(self, module)
    _qmm.Qwen3_5MoeForConditionalGeneration._init_weights = _patched_init
    print('[patch] Qwen3_5MoeGatedDeltaNet init no-op installed')
except Exception as e:
    print(f'[patch] could not install GatedDeltaNet patch: {type(e).__name__}: {e}')

# --- 4. Configure backbone --------------------------------------------------
os.environ['QWEN_MODEL_PATH']     = str(QWEN_DIR)
os.environ['QWEN_DTYPE']          = 'bf16'
os.environ['QWEN_RUNTIME']        = 'vllm' if VLLM_AVAILABLE else 'transformers'
os.environ['QWEN_FRAME_UPSCALE']  = '4'
os.environ['QWEN_TEMPERATURE']    = '0.7'
os.environ['QWEN_TOP_P']          = '0.9'
os.environ['QWEN_MAX_NEW_TOKENS'] = '24'
os.environ['QWEN_HISTORY_LEN']    = '8'
os.environ['QWEN_DEBUG_PROMPTS']  = '0'  # flip to 1 if you need /tmp/qwen_trace.log

# --- 5. Setup arc-agi SDK in OFFLINE mode -----------------------------------
os.environ['OPERATION_MODE'] = 'offline'
_ENV_CANDS = [
    Path('/kaggle/input/arc-prize-2026-arc-agi-3/environment_files'),
    Path('/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files'),
]
ENV_DIR = next((c for c in _ENV_CANDS if c.exists()), None)
if ENV_DIR is None:
    for p in Path('/kaggle/input').rglob('environment_files'):
        if p.is_dir():
            ENV_DIR = p
            break
if ENV_DIR is not None:
    os.environ['ENVIRONMENTS_DIR'] = str(ENV_DIR)
    print(f'ENV_DIR = {ENV_DIR} ({sum(1 for _ in ENV_DIR.iterdir())} games)')

# --- 6. Run QwenPolicyAgent on each public game ----------------------------
from arc_agi import Arcade, OperationMode  # noqa: E402
from agents import GameAction, GameState  # noqa: E402
from agents.qwen_policy_agent import QwenPolicyAgent  # noqa: E402

GAMES = os.environ.get('SMOKE_GAMES', 'ls20,vc33,ft09').split(',')
MAX_ACTIONS = int(os.environ.get('MAX_ACTIONS', 200))
all_results = {'games': []}

agent = QwenPolicyAgent(seed=0)
# Pre-load model once; reuse across games
print('Pre-loading model...')
_t0 = time.time()
agent.backbone._ensure_loaded()
print(f'  model loaded in {time.time() - _t0:.1f}s, runtime={agent.backbone.actual_runtime}')

arcade = Arcade(operation_mode=OperationMode.OFFLINE,
                environments_dir=os.environ.get('ENVIRONMENTS_DIR', 'environment_files'))

for game_id in GAMES:
    game_id = game_id.strip()
    if not game_id:
        continue
    print(f'\n=== {game_id} (max_actions={MAX_ACTIONS}) ===')
    env = arcade.make(game_id)
    frame = env.observation_space
    agent2 = QwenPolicyAgent(seed=0, backbone=agent.backbone)
    history_log = []
    n_changed = 0
    n_steps = 0
    t0 = time.time()
    for step in range(MAX_ACTIONS):
        if frame.state in (GameState.WIN, GameState.GAME_OVER):
            print(f'  [step {step}] terminal {frame.state}; break')
            break
        ts = time.time()
        action = agent2.choose_action(frame)
        dt = time.time() - ts
        n_steps += 1
        data = {}
        ad = getattr(action, 'action_data', None)
        if ad is not None:
            try: data = ad.model_dump()
            except Exception: data = getattr(action, '_data', {}) or {}
        prev_levels = getattr(frame, 'levels_completed', 0)
        next_frame = env.step(action, data=data, reasoning=None)
        if next_frame is None:
            print('  env.step returned None; break')
            break
        # Detect frame change
        try:
            import numpy as _np
            prev_a = _np.asarray(frame.frame[0])
            next_a = _np.asarray(next_frame.frame[0])
            changed = not _np.array_equal(prev_a, next_a)
        except Exception:
            changed = (frame.frame != next_frame.frame)
        if changed:
            n_changed += 1
        history_log.append({
            'step': step, 'action': action.name,
            'state': str(frame.state), 'changed': bool(changed),
            'levels_completed': getattr(next_frame, 'levels_completed', prev_levels),
            'dt': round(dt, 3),
        })
        if step < 20 or step % 20 == 0:
            print(f'  [step {step}] {action.name}  changed={changed}  '
                  f'level={getattr(next_frame, "levels_completed", prev_levels)}  '
                  f'dt={dt:.2f}s')
        frame = next_frame
    elapsed = time.time() - t0
    levels = getattr(frame, 'levels_completed', 0)
    summary = {
        'game': game_id,
        'actions_taken': n_steps,
        'levels_completed': levels,
        'win_levels': getattr(frame, 'win_levels', 0),
        'final_state': str(frame.state),
        'change_rate': round(n_changed / max(n_steps, 1), 3),
        'wall_clock_s': round(elapsed, 1),
        'mean_s_per_action': round(elapsed / max(n_steps, 1), 3),
        'graph_nodes': len(agent2.graph.nodes),
    }
    print(f'  SUMMARY: {summary}')
    all_results['games'].append({**summary, 'history': history_log})

all_results['runtime'] = agent.backbone.actual_runtime
all_results['vllm_available'] = VLLM_AVAILABLE
all_results['totals'] = {
    'total_levels': sum(g['levels_completed'] for g in all_results['games']),
    'games_with_at_least_one_level': sum(1 for g in all_results['games'] if g['levels_completed'] >= 1),
    'total_actions': sum(g['actions_taken'] for g in all_results['games']),
    'mean_s_per_action_overall': round(
        sum(g['wall_clock_s'] for g in all_results['games']) / max(sum(g['actions_taken'] for g in all_results['games']), 1),
        3,
    ),
}
with open('/kaggle/working/qwen_policy_dev.json', 'w') as fh:
    json.dump(all_results, fh, indent=2)
print('\n=== TOTALS ===')
print(json.dumps(all_results['totals'], indent=2))
print('saved /kaggle/working/qwen_policy_dev.json')
